In [6]:
# ============================================================
# CELLULE 1 — Imports + Paramètres
# ============================================================
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn as nn

# Vérification GPU
if torch.cuda.is_available():
    print(f"✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("⚠️  CPU uniquement")

# Chemins
DATA_DIR = '/kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final'
CSV_PATH = '/kaggle/working/labels.csv'

# Paramètres
IMG_SIZE   = 224
BATCH_SIZE = 32
SEED       = 42

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"\n📁 Data dir : {DATA_DIR}")
print(f"📁 CSV path : {CSV_PATH}")

✅ GPU : Tesla T4
   Mémoire : 14.6 GB

📁 Data dir : /kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final
📁 CSV path : /kaggle/working/labels.csv


In [1]:
import os
print(os.listdir("/kaggle/working"))

['.virtual_documents', 'resnet50_phase1_best.pth', 'runs', 'models_final', 'yolo26n.pt', 'labels.csv', 'resnet50_phase2_best.pth', 'yolov8m-cls.pt', 'efficientnet_phase1_best.pth', 'state.db', 'efficientnet_phase2_best.pth']


In [2]:
import os
print(os.listdir("/kaggle/working/runs"))

['classify', 'yolov8_phase1', 'yolov8_phase2']


In [2]:
# ============================================================
# CELLULE 2 — Nettoyage
# ============================================================
import shutil

# Supprimer labels.csv
if os.path.exists('/kaggle/working/labels.csv'):
    os.remove('/kaggle/working/labels.csv')
    print("✅ Supprimé : labels.csv")

# Supprimer runs (YOLOv8)
if os.path.exists('/kaggle/working/runs'):
    shutil.rmtree('/kaggle/working/runs')
    print("✅ Supprimé : runs/")

# Supprimer fichiers .pth et .pt
for f in os.listdir('/kaggle/working/'):
    if f.endswith(('.pth', '.pt', '.db')):
        os.remove(f'/kaggle/working/{f}')
        print(f"✅ Supprimé : {f}")

print("\n✅ Nettoyage terminé !")

✅ Supprimé : labels.csv
✅ Supprimé : state.db

✅ Nettoyage terminé !


In [7]:
# ============================================================
# RECRÉATION labels.csv depuis les dossiers
# ============================================================
import pandas as pd
import os

label_to_groupe = {
    'tail_wagging'      : 'normal',
    'playing'           : 'normal',
    'sitting'           : 'normal',
    'standing'          : 'normal',
    'eating'            : 'normal',
    'lying'             : 'normal',
    'restlessness'      : 'anormal',
    'paralysis'         : 'anormal',
    'incoordination'    : 'anormal',
    'digging'           : 'anormal',
    'barking'           : 'anormal',
    'hyper_salivation'  : 'anormal',
    'bone_in_throat'    : 'anormal',
    'dropped_jaw'       : 'anormal',
    'sudden_aggression' : 'anormal',
    'seizure'           : 'anormal',
}

records = []

for split in ['train', 'val', 'test']:
    split_dir = os.path.join(DATA_DIR, split)
    for label in os.listdir(split_dir):
        label_dir = os.path.join(split_dir, label)
        if not os.path.isdir(label_dir):
            continue
        for fname in os.listdir(label_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({
                    'filepath' : os.path.join(split, label, fname),
                    'label'    : label,
                    'groupe'   : label_to_groupe.get(label, 'inconnu'),
                    'split'    : split
                })

df = pd.DataFrame(records)

# Sauvegarder dans /kaggle/working/
CSV_PATH = '/kaggle/working/labels.csv'
df.to_csv(CSV_PATH, index=False)

print(f"✅ labels.csv recréé : {len(df)} lignes\n")
print("📊 Distribution par split et groupe :")
print(df.groupby(['split', 'groupe']).size().unstack(fill_value=0))
print("\n📊 Distribution par label :")
print(df.groupby(['split', 'label']).size().unstack(fill_value=0))

✅ labels.csv recréé : 6400 lignes

📊 Distribution par split et groupe :
groupe  anormal  normal
split                  
test        400     240
train      3200    1920
val         400     240

📊 Distribution par label :
label  barking  bone_in_throat  digging  dropped_jaw  eating  \
split                                                          
test        40              40       40           40      40   
train      320             320      320          320     320   
val         40              40       40           40      40   

label  hyper_salivation  incoordination  lying  paralysis  playing  \
split                                                                
test                 40              40     40         40       40   
train               320             320    320        320      320   
val                  40              40     40         40       40   

label  restlessness  seizure  sitting  standing  sudden_aggression  \
split                                 

In [8]:
# ============================================================
# CELLULE 4 — Dataset PyTorch
# ============================================================
from torch.utils.data import Dataset
import torchvision.transforms as transforms

# Liste des 16 labels triés
LABELS    = sorted(df['label'].unique().tolist())
LABEL2ID  = {label: i for i, label in enumerate(LABELS)}
ID2LABEL  = {i: label for label, i in LABEL2ID.items()}

print(f"📋 {len(LABELS)} labels :")
for i, label in enumerate(LABELS):
    groupe = df[df['label'] == label]['groupe'].iloc[0]
    print(f"  {i:2d} → {label:25s} ({groupe})")

# Transformations
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class BehaviorDataset(Dataset):
    def __init__(self, df, data_dir, split, transform=None):
        self.df        = df[df['split'] == split].reset_index(drop=True)
        self.data_dir  = data_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        path  = os.path.join(self.data_dir, row['filepath'])
        label = LABEL2ID[row['label']]
        img   = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label

# Créer les datasets
train_dataset = BehaviorDataset(df, DATA_DIR, 'train', train_transform)
val_dataset   = BehaviorDataset(df, DATA_DIR, 'val',   val_transform)
test_dataset  = BehaviorDataset(df, DATA_DIR, 'test',  val_transform)

# Créer les dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2)

print(f"\n✅ Datasets créés :")
print(f"   train : {len(train_dataset)} images")
print(f"   val   : {len(val_dataset)} images")
print(f"   test  : {len(test_dataset)} images")

📋 16 labels :
   0 → barking                   (anormal)
   1 → bone_in_throat            (anormal)
   2 → digging                   (anormal)
   3 → dropped_jaw               (anormal)
   4 → eating                    (normal)
   5 → hyper_salivation          (anormal)
   6 → incoordination            (anormal)
   7 → lying                     (normal)
   8 → paralysis                 (anormal)
   9 → playing                   (normal)
  10 → restlessness              (anormal)
  11 → seizure                   (anormal)
  12 → sitting                   (normal)
  13 → standing                  (normal)
  14 → sudden_aggression         (anormal)
  15 → tail_wagging              (normal)

✅ Datasets créés :
   train : 5120 images
   val   : 640 images
   test  : 640 images


In [9]:
# ============================================================
# CELLULE 5 — Création EfficientNetB0
# ============================================================
import torchvision.models as models

def create_efficientnet(num_classes=16, dropout=0.5):
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')
    
    # Geler le backbone (Phase 1)
    for param in model.parameters():
        param.requires_grad = False
    
    # Remplacer le classifier
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    return model

model_eff = create_efficientnet(num_classes=16).to(device)

total_params     = sum(p.numel() for p in model_eff.parameters())
trainable_params = sum(p.numel() for p in model_eff.parameters()
                       if p.requires_grad)

print("✅ EfficientNetB0 créé")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 130MB/s] 


✅ EfficientNetB0 créé
   Total params     : 4,028,044
   Trainable params : 20,496


In [6]:
# ============================================================
# CELLULE 5— Fonction d'entraînement + Early Stopping
# ============================================================
import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

def train_model(model, train_loader, val_loader,
                epochs, lr, label_smoothing=0.1,
                weight_decay=1e-4, phase=1,
                patience=5, model_name='model'):

    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay
    )

    scheduler = StepLR(optimizer, step_size=5, gamma=0.5)

    best_val_acc    = 0.0
    patience_counter = 0
    history         = {'train_loss': [], 'train_acc': [],
                       'val_loss':   [], 'val_acc':   []}

    print(f"🚀 Phase {phase} — lr={lr} | epochs={epochs} | "
          f"label_smoothing={label_smoothing} | patience={patience}\n")

    for epoch in range(epochs):

        # ── TRAIN ──────────────────────────────────────────
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss    += loss.item()
            preds          = outputs.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total   += labels.size(0)

        # ── VALIDATION ─────────────────────────────────────
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                outputs     = model(imgs)
                loss        = criterion(outputs, labels)
                val_loss   += loss.item()
                preds       = outputs.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total   += labels.size(0)

        # ── Métriques ──────────────────────────────────────
        t_loss = train_loss / len(train_loader)
        t_acc  = train_correct / train_total * 100
        v_loss = val_loss / len(val_loader)
        v_acc  = val_correct / val_total * 100

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        # ── Early Stopping ─────────────────────────────────
        if v_acc > best_val_acc:
            best_val_acc     = v_acc
            patience_counter = 0
            torch.save(model.state_dict(),
                       f'/kaggle/working/{model_name}_phase{phase}_best.pth')
            saved = "💾"
        else:
            patience_counter += 1
            saved = f"⏳ {patience_counter}/{patience}"

        scheduler.step()

        print(f"  Epoch {epoch+1:2d}/{epochs} | "
              f"Train Loss: {t_loss:.4f} Acc: {t_acc:.1f}% | "
              f"Val Loss: {v_loss:.4f} Acc: {v_acc:.1f}% {saved}")

        # Stop si patience dépassée
        if patience_counter >= patience:
            print(f"\n Early stopping à epoch {epoch+1} "
                  f"— pas d'amélioration depuis {patience} epochs")
            break

    print(f"\n✅ Phase {phase} terminée — "
          f"Meilleure Val Acc: {best_val_acc:.1f}%")
    return history

print("✅ Fonction d'entraînement avec Early Stopping définie")

✅ Fonction d'entraînement avec Early Stopping définie


In [7]:
# ============================================================
# CELLULE 6 — Entraînement EfficientNet Phase 1
# ============================================================

history_eff_p1 = train_model(
    model       = model_eff,
    train_loader= train_loader,
    val_loader  = val_loader,
    epochs      = 10,
    lr          = 0.001,
    phase       = 1,
    model_name   = 'efficientnet'
)

🚀 Phase 1 — lr=0.001 | epochs=10 | label_smoothing=0.1 | patience=5

  Epoch  1/10 | Train Loss: 1.9378 Acc: 51.2% | Val Loss: 1.4607 Acc: 70.2% 💾
  Epoch  2/10 | Train Loss: 1.4446 Acc: 67.3% | Val Loss: 1.2467 Acc: 75.9% 💾
  Epoch  3/10 | Train Loss: 1.3216 Acc: 72.7% | Val Loss: 1.1622 Acc: 78.3% 💾
  Epoch  4/10 | Train Loss: 1.2531 Acc: 74.5% | Val Loss: 1.1072 Acc: 81.6% 💾
  Epoch  5/10 | Train Loss: 1.2206 Acc: 75.8% | Val Loss: 1.0654 Acc: 82.5% 💾
  Epoch  6/10 | Train Loss: 1.1879 Acc: 77.6% | Val Loss: 1.0499 Acc: 83.1% 💾
  Epoch  7/10 | Train Loss: 1.1787 Acc: 77.5% | Val Loss: 1.0485 Acc: 83.1% ⏳ 1/5
  Epoch  8/10 | Train Loss: 1.1814 Acc: 78.2% | Val Loss: 1.0285 Acc: 83.9% 💾
  Epoch  9/10 | Train Loss: 1.1614 Acc: 78.3% | Val Loss: 1.0206 Acc: 84.2% 💾
  Epoch 10/10 | Train Loss: 1.1500 Acc: 79.3% | Val Loss: 1.0117 Acc: 84.5% 💾

✅ Phase 1 terminée — Meilleure Val Acc: 84.5%


In [8]:
# ============================================================
# CELLULE 7 — EfficientNet Phase 2 (Fine-tuning)
# ============================================================

# Charger le meilleur modèle de Phase 1
model_eff.load_state_dict(
    torch.load('/kaggle/working/efficientnet_phase1_best.pth'))  # ← nom correct

# Dégeler TOUT le modèle
for param in model_eff.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model_eff.parameters()
                   if p.requires_grad)
print(f"✅ Backbone dégelé — {total_params:,} paramètres entraînables\n")

history_eff_p2 = train_model(
    model        = model_eff,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 30,
    lr           = 0.0001,
    phase        = 2,
    patience     = 7,
    model_name   = 'efficientnet'
)

✅ Backbone dégelé — 4,028,044 paramètres entraînables

🚀 Phase 2 — lr=0.0001 | epochs=30 | label_smoothing=0.1 | patience=7

  Epoch  1/30 | Train Loss: 1.0550 Acc: 82.7% | Val Loss: 0.8484 Acc: 89.5% 💾
  Epoch  2/30 | Train Loss: 0.9085 Acc: 89.0% | Val Loss: 0.8024 Acc: 92.2% 💾
  Epoch  3/30 | Train Loss: 0.8557 Acc: 90.8% | Val Loss: 0.7770 Acc: 92.8% 💾
  Epoch  4/30 | Train Loss: 0.8122 Acc: 92.3% | Val Loss: 0.7720 Acc: 93.1% 💾
  Epoch  5/30 | Train Loss: 0.7939 Acc: 92.9% | Val Loss: 0.7570 Acc: 93.3% 💾
  Epoch  6/30 | Train Loss: 0.7748 Acc: 93.9% | Val Loss: 0.7589 Acc: 93.1% ⏳ 1/7
  Epoch  7/30 | Train Loss: 0.7558 Acc: 94.4% | Val Loss: 0.7486 Acc: 93.1% ⏳ 2/7
  Epoch  8/30 | Train Loss: 0.7536 Acc: 94.1% | Val Loss: 0.7512 Acc: 93.1% ⏳ 3/7
  Epoch  9/30 | Train Loss: 0.7465 Acc: 94.3% | Val Loss: 0.7494 Acc: 93.0% ⏳ 4/7
  Epoch 10/30 | Train Loss: 0.7364 Acc: 94.9% | Val Loss: 0.7455 Acc: 93.4% 💾
  Epoch 11/30 | Train Loss: 0.7338 Acc: 94.7% | Val Loss: 0.7472 Acc: 92.8% ⏳ 1

In [13]:
# ============================================================
# CELLULE 8 — Modèle ResNet50
# ============================================================
import torchvision.models as models
import torch.nn as nn

def create_resnet50(num_classes=16, dropout=0.5):
    model = models.resnet50(weights='IMAGENET1K_V1')
    
    # Geler le backbone (Phase 1)
    for param in model.parameters():
        param.requires_grad = False
    
    # Remplacer le classifier par notre tête
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=dropout),
        nn.Linear(in_features, num_classes)
    )
    
    return model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_res = create_resnet50(num_classes=16).to(device)

# Vérification
total_params     = sum(p.numel() for p in model_res.parameters())
trainable_params = sum(p.numel() for p in model_res.parameters()
                       if p.requires_grad)

print("✅ ResNet50 créé")
print(f"   Total params     : {total_params:,}")
print(f"   Trainable params : {trainable_params:,} (Phase 1 — tête uniquement)")
print(f"   Classes          : 16 comportements")
print(f"   Dropout          : 0.5")

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 168MB/s] 


✅ ResNet50 créé
   Total params     : 23,540,816
   Trainable params : 32,784 (Phase 1 — tête uniquement)
   Classes          : 16 comportements
   Dropout          : 0.5


In [10]:
history_res_p1 = train_model(
    model        = model_res,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 25,
    lr           = 0.001,
    phase        = 1,
    patience     = 5,
    model_name   = 'resnet50'
)

🚀 Phase 1 — lr=0.001 | epochs=25 | label_smoothing=0.1 | patience=5

  Epoch  1/25 | Train Loss: 1.9452 Acc: 44.2% | Val Loss: 1.4079 Acc: 69.1% 💾
  Epoch  2/25 | Train Loss: 1.4774 Acc: 62.7% | Val Loss: 1.3535 Acc: 70.8% 💾
  Epoch  3/25 | Train Loss: 1.3924 Acc: 66.4% | Val Loss: 1.2162 Acc: 75.5% 💾
  Epoch  4/25 | Train Loss: 1.3543 Acc: 69.6% | Val Loss: 1.2062 Acc: 76.6% 💾
  Epoch  5/25 | Train Loss: 1.3351 Acc: 70.5% | Val Loss: 1.1801 Acc: 78.9% 💾
  Epoch  6/25 | Train Loss: 1.2856 Acc: 72.0% | Val Loss: 1.1547 Acc: 78.1% ⏳ 1/5
  Epoch  7/25 | Train Loss: 1.2718 Acc: 72.5% | Val Loss: 1.1521 Acc: 78.0% ⏳ 2/5
  Epoch  8/25 | Train Loss: 1.2787 Acc: 72.7% | Val Loss: 1.1200 Acc: 81.1% 💾
  Epoch  9/25 | Train Loss: 1.2661 Acc: 73.4% | Val Loss: 1.1383 Acc: 79.7% ⏳ 1/5
  Epoch 10/25 | Train Loss: 1.2600 Acc: 73.7% | Val Loss: 1.1280 Acc: 78.9% ⏳ 2/5
  Epoch 11/25 | Train Loss: 1.2430 Acc: 73.8% | Val Loss: 1.1102 Acc: 80.3% ⏳ 3/5
  Epoch 12/25 | Train Loss: 1.2426 Acc: 74.1% | Val L

In [11]:
# ============================================================
# CELLULE 10 — ResNet50 Phase 2 (Fine-tuning)
# ============================================================

model_res.load_state_dict(
    torch.load('/kaggle/working/resnet50_phase1_best.pth'))

for param in model_res.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model_res.parameters()
                   if p.requires_grad)
print(f"✅ Backbone dégelé — {total_params:,} paramètres entraînables\n")

history_res_p2 = train_model(
    model        = model_res,
    train_loader = train_loader,
    val_loader   = val_loader,
    epochs       = 30,
    lr           = 0.0001,
    phase        = 2,
    patience     = 7,
    model_name   = 'resnet50'
)

✅ Backbone dégelé — 23,540,816 paramètres entraînables

🚀 Phase 2 — lr=0.0001 | epochs=30 | label_smoothing=0.1 | patience=7

  Epoch  1/30 | Train Loss: 1.1391 Acc: 79.4% | Val Loss: 0.9357 Acc: 88.9% 💾
  Epoch  2/30 | Train Loss: 0.9124 Acc: 89.6% | Val Loss: 0.8496 Acc: 91.6% 💾
  Epoch  3/30 | Train Loss: 0.8451 Acc: 92.2% | Val Loss: 0.7969 Acc: 92.7% 💾
  Epoch  4/30 | Train Loss: 0.8038 Acc: 93.1% | Val Loss: 0.7793 Acc: 91.7% ⏳ 1/7
  Epoch  5/30 | Train Loss: 0.7795 Acc: 93.6% | Val Loss: 0.7462 Acc: 93.1% 💾
  Epoch  6/30 | Train Loss: 0.7495 Acc: 94.3% | Val Loss: 0.7387 Acc: 93.1% ⏳ 1/7
  Epoch  7/30 | Train Loss: 0.7262 Acc: 94.9% | Val Loss: 0.7380 Acc: 93.6% 💾
  Epoch  8/30 | Train Loss: 0.7237 Acc: 94.9% | Val Loss: 0.7318 Acc: 93.1% ⏳ 1/7
  Epoch  9/30 | Train Loss: 0.7165 Acc: 94.8% | Val Loss: 0.7509 Acc: 92.8% ⏳ 2/7
  Epoch 10/30 | Train Loss: 0.7111 Acc: 95.2% | Val Loss: 0.7329 Acc: 92.8% ⏳ 3/7
  Epoch 11/30 | Train Loss: 0.6972 Acc: 95.5% | Val Loss: 0.7326 Acc: 92.5

In [12]:
# ============================================================
# CELLULE 11 — Installation YOLOv8
# ============================================================
!pip install ultralytics -q

from ultralytics import YOLO
import torch

print(f"✅ GPU : {torch.cuda.get_device_name(0)}")

# Créer le modèle YOLOv8m-cls
model_yolo = YOLO('yolov8m-cls.pt')

print("✅ YOLOv8m-cls chargé")
print(f"   Classes : 16 comportements")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.3 MB/s eta 0:00:00a 0:00:01
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ GPU : Tesla T4
✅ YOLOv8m-cls chargé
   Classes : 16 comportements


In [13]:
# ============================================================
# CELLULE 12 — YOLOv8 Phase 1
# ============================================================

print(" Phase 1 — YOLOv8m-cls\n")

results_yolo_p1 = model_yolo.train(
    data     = DATA_DIR,
    epochs   = 15,
    imgsz    = 224,
    batch    = 32,
    device   = 0,
    patience = 5,
    save     = True,
    plots    = True,
    workers  = 2,
    augment  = False,
    lr0      = 0.001,
    lrf      = 0.01,
    dropout  = 0.5,
    project  = '/kaggle/working/runs',
    name     = 'yolov8_phase1'
)

print("✅ Phase 1 YOLOv8 terminée !")

 Phase 1 — YOLOv8m-cls

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.5, dynamic=False, embed=None, end2end=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8_phase1, nbs=64, nms=False, opset=None

In [14]:
# ============================================================
# CELLULE 13 — YOLOv8 Phase 2 (Fine-tuning)
# ============================================================

print("🚀 Phase 2 — YOLOv8m-cls\n")

model_yolo2 = YOLO('/kaggle/working/runs/yolov8_phase1/weights/best.pt')

results_yolo_p2 = model_yolo2.train(
    data     = DATA_DIR,
    epochs   = 30,
    imgsz    = 224,
    batch    = 32,
    device   = 0,
    patience = 7,
    save     = True,
    plots    = True,
    workers  = 2,
    augment  = False,
    lr0      = 0.0001,
    lrf      = 0.01,
    dropout  = 0.5,
    project  = '/kaggle/working/runs',
    name     = 'yolov8_phase2'
)

print("✅ Phase 2 YOLOv8 terminée !")

🚀 Phase 2 — YOLOv8m-cls

Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.5, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/yolov8_phase1/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8

In [15]:
# ============================================================
# CELLULE 14 — Évaluation EfficientNet + ResNet sur Test Set
# ============================================================
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_model(model, test_loader, model_name):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in test_loader:
            imgs   = imgs.to(device)
            outputs = model(imgs)
            preds   = outputs.argmax(dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    acc = sum(p == l for p, l in
              zip(all_preds, all_labels)) / len(all_labels) * 100

    print(f"\n{'='*50}")
    print(f"📊 {model_name} — Test Accuracy : {acc:.1f}%")
    print(f"{'='*50}")
    print(classification_report(all_labels, all_preds,
                                target_names=LABELS))
    return all_preds, all_labels

# Charger les meilleurs modèles
model_eff.load_state_dict(
    torch.load('/kaggle/working/efficientnet_phase2_best.pth'))
model_res.load_state_dict(
    torch.load('/kaggle/working/resnet50_phase2_best.pth'))

# Évaluation
preds_eff, labels_test = evaluate_model(
    model_eff, test_loader, 'EfficientNetB0')
preds_res, _           = evaluate_model(
    model_res, test_loader, 'ResNet50')


📊 EfficientNetB0 — Test Accuracy : 93.4%
                   precision    recall  f1-score   support

          barking       1.00      0.95      0.97        40
   bone_in_throat       0.84      0.93      0.88        40
          digging       1.00      1.00      1.00        40
      dropped_jaw       0.79      0.75      0.77        40
           eating       1.00      1.00      1.00        40
 hyper_salivation       0.87      0.82      0.85        40
   incoordination       0.86      0.75      0.80        40
            lying       1.00      1.00      1.00        40
        paralysis       0.95      0.95      0.95        40
          playing       1.00      1.00      1.00        40
     restlessness       0.74      0.93      0.82        40
          seizure       1.00      0.97      0.99        40
          sitting       1.00      1.00      1.00        40
         standing       1.00      1.00      1.00        40
sudden_aggression       0.95      0.90      0.92        40
     tail_wag

In [16]:
# ============================================================
# CELLULE 15 — Évaluation YOLOv8 sur Test Set
# ============================================================
from ultralytics import YOLO

model_yolo_test = YOLO('/kaggle/working/runs/yolov8_phase2/weights/best.pt')

results = model_yolo_test.val(
    data    = DATA_DIR,
    split   = 'test',
    imgsz   = 224,
    batch   = 32,
    device  = 0,
)

print(f"\n✅ YOLOv8 Test Accuracy : {results.top1:.3f}")


Ultralytics 8.4.37 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8m-cls summary (fused): 42 layers, 15,783,152 parameters, 0 gradients, 41.7 GFLOPs
train: /kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final/train... found 5120 images in 16 classes ✅ 
val: /kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final/val... found 640 images in 16 classes ✅ 
test: /kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final/test... found 640 images in 16 classes ✅ 
test: Fast image access ✅ (ping: 1.0±0.0 ms, read: 13.9±2.5 MB/s, size: 14.6 KB)
test: Scanning /kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final/test... 640 images, 0 corrupt: 100% ━━━━━━━━━━━━ 640/640 1.2Kit/s 0.5s0.0ss
WARNING ⚠️ test: Cache directory /kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final is not writable, cache not saved.
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 20/20 12.5it/s 1.6s0.1s
                   all      0.919      0.998
Sp

In [17]:
# ============================================================
# CELLULE 15B — Rapport détaillé YOLOv8 par classe
# ============================================================
from sklearn.metrics import classification_report
import numpy as np
from PIL import Image
import torch

model_yolo_test = YOLO('/kaggle/working/runs/yolov8_phase2/weights/best.pt')

all_preds  = []
all_labels = []

print("🔍 Extraction des prédictions YOLOv8...\n")

for split_label in LABELS:
    test_dir = os.path.join(DATA_DIR, 'test', split_label)
    label_id = LABEL2ID[split_label]

    imgs_files = [f for f in os.listdir(test_dir)
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))]

    for fname in imgs_files:
        fpath  = os.path.join(test_dir, fname)
        result = model_yolo_test(fpath, verbose=False)
        pred   = result[0].probs.top1
        all_preds.append(pred)
        all_labels.append(label_id)

acc = sum(p == l for p, l in
          zip(all_preds, all_labels)) / len(all_labels) * 100

print(f"📊 YOLOv8 — Test Accuracy : {acc:.1f}%")
print(f"{'='*50}")
print(classification_report(all_labels, all_preds,
                            target_names=LABELS))

🔍 Extraction des prédictions YOLOv8...

📊 YOLOv8 — Test Accuracy : 91.9%
                   precision    recall  f1-score   support

          barking       1.00      0.90      0.95        40
   bone_in_throat       0.84      0.80      0.82        40
          digging       0.98      1.00      0.99        40
      dropped_jaw       0.88      0.75      0.81        40
           eating       1.00      1.00      1.00        40
 hyper_salivation       0.80      0.88      0.83        40
   incoordination       0.90      0.65      0.75        40
            lying       0.97      0.97      0.97        40
        paralysis       0.95      0.97      0.96        40
          playing       1.00      0.88      0.93        40
     restlessness       0.74      1.00      0.85        40
          seizure       1.00      0.97      0.99        40
          sitting       0.98      1.00      0.99        40
         standing       1.00      1.00      1.00        40
sudden_aggression       0.90      0.93   

## Les 3 modèles montrent des résultats cohérents entre validation et test avec des écarts inférieurs à 1.1%, ce qui confirme l'absence d'overfitting. Les classes difficiles comme dropped_jaw et incoordination sont difficiles dans les 3 modèles, ce qui indique une similarité visuelle intrinsèque entre ces comportements plutôt qu'un problème de modélisation. 

Les classes difficiles comme dropped_jaw et incoordination présentent une similarité visuelle intrinsèque qui limite la précision individuelle. Cependant, notre système de Late Fusion compense ces limitations en combinant les probabilités des 3 modèles et des 3 modalités (comportement, émotion, audio), ce qui améliore la robustesse du diagnostic final.

In [18]:
# ============================================================
# CELLULE 16 — Copier dans Output Kaggle
# ============================================================
import shutil, os

os.makedirs('/kaggle/working/models_final', exist_ok=True)

shutil.copy('/kaggle/working/efficientnet_phase2_best.pth',
            '/kaggle/working/models_final/efficientnet_best.pth')

shutil.copy('/kaggle/working/resnet50_phase2_best.pth',
            '/kaggle/working/models_final/resnet50_best.pth')

shutil.copy('/kaggle/working/runs/yolov8_phase2/weights/best.pt',
            '/kaggle/working/models_final/yolov8_best.pt')

print("✅ Modèles copiés dans Output :")
for f in os.listdir('/kaggle/working/models_final/'):
    size = os.path.getsize(f'/kaggle/working/models_final/{f}')
    print(f"  📦 {f} — {size/(1024*1024):.1f} MB")

✅ Modèles copiés dans Output :
  📦 efficientnet_best.pth — 15.7 MB
  📦 resnet50_best.pth — 90.1 MB
  📦 yolov8_best.pt — 30.2 MB


In [4]:
import pandas as pd
print(pd.read_csv("/kaggle/working/labels.csv")['label'].unique())

['sitting' 'playing' 'bone_in_throat' 'lying' 'standing' 'digging'
 'tail_wagging' 'hyper_salivation' 'sudden_aggression' 'seizure'
 'paralysis' 'restlessness' 'incoordination' 'eating' 'dropped_jaw'
 'barking']


In [17]:
import torchvision.models as models
from ultralytics import YOLO

NUM_CLASSES_BEHAV = 16

# Charger EfficientNetB0
model_eff = models.efficientnet_b0(weights=None)
model_eff.classifier = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_eff.classifier[1].in_features, NUM_CLASSES_BEHAV)
)
model_eff.load_state_dict(torch.load("/kaggle/working/efficientnet_phase2_best.pth"))
model_eff = model_eff.to(device)
model_eff.eval()
print("✅ EfficientNetB0 chargé")

# Charger ResNet50
model_res = models.resnet50(weights=None)
model_res.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model_res.fc.in_features, NUM_CLASSES_BEHAV)
)
model_res.load_state_dict(torch.load("/kaggle/working/resnet50_phase2_best.pth"))
model_res = model_res.to(device)
model_res.eval()
print("✅ ResNet50 chargé")


model_yolo = YOLO("/kaggle/working/models_final/yolov8_best.pt")
print("✅ YOLOv8 chargé")

✅ EfficientNetB0 chargé
✅ ResNet50 chargé
✅ YOLOv8 chargé


In [19]:
import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader

# Charger labels.csv
df = pd.read_csv("/kaggle/working/labels.csv")
test_df = df[df["split"] == "test"].reset_index(drop=True)

# Mapping normal/anormal
NORMAL_CLASSES = ['sitting', 'playing', 'lying', 'standing', 
                  'digging', 'tail_wagging', 'eating', 'barking']
ANORMAL_CLASSES = ['bone_in_throat', 'hyper_salivation', 'sudden_aggression', 
                   'seizure', 'paralysis', 'restlessness', 
                   'incoordination', 'dropped_jaw']

CLASSES = sorted(test_df['label'].unique().tolist())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
print(f"Classes : {CLASSES}")
print(f"Test set : {len(test_df)} images")
print("✅ OK")

Classes : ['barking', 'bone_in_throat', 'digging', 'dropped_jaw', 'eating', 'hyper_salivation', 'incoordination', 'lying', 'paralysis', 'playing', 'restlessness', 'seizure', 'sitting', 'standing', 'sudden_aggression', 'tail_wagging']
Test set : 640 images
✅ OK


In [20]:
from torchvision import transforms
from torch.utils.data import Dataset
from PIL import Image

# Dataset
class BehaviorDataset(Dataset):
    def __init__(self, df, data_dir, transform):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(f"{self.data_dir}/{row['filepath']}").convert("RGB")
        label = CLASS_TO_IDX[row['label']]
        return self.transform(img), label

val_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

test_loader = DataLoader(
    BehaviorDataset(test_df, "/kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final", val_tf),
    batch_size=32, shuffle=False, num_workers=2
)

# Extraire probabilités EfficientNetB0
all_probs_eff = []
all_labels    = []

model_eff.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = torch.softmax(model_eff(imgs), dim=1)
        all_probs_eff.extend(out.cpu().numpy())
        all_labels.extend(labels.numpy())

all_probs_eff = np.array(all_probs_eff)
all_labels    = np.array(all_labels)

print(f"✅ EfficientNetB0 probs extraites : {all_probs_eff.shape}")

✅ EfficientNetB0 probs extraites : (640, 16)


In [21]:
# Extraire probabilités ResNet50
all_probs_res = []

model_res.eval()
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        out  = torch.softmax(model_res(imgs), dim=1)
        all_probs_res.extend(out.cpu().numpy())

all_probs_res = np.array(all_probs_res)
print(f"✅ ResNet50 probs extraites : {all_probs_res.shape}")

✅ ResNet50 probs extraites : (640, 16)


In [26]:
import glob

# Collecter tous les chemins d'images du test set
test_images = glob.glob("/kaggle/input/datasets/zeinebbayoudh7/behavior-v2-final/final/test/*/*.jpg")
print(f"Total images test : {len(test_images)}")

# Extraire probabilités YOLOv8
all_probs_yolo = []

results = model_yolo.predict(
    source=test_images,
    imgsz=224,
    batch=32,
    verbose=False
)

for r in results:
    probs = r.probs.data.cpu().numpy()
    all_probs_yolo.append(probs)

all_probs_yolo = np.array(all_probs_yolo)
print(f"✅ YOLOv8 probs extraites : {all_probs_yolo.shape}")

Total images test : 640
✅ YOLOv8 probs extraites : (640, 16)


In [27]:
# Fusion des 3 modèles (moyenne pondérée)
# EfficientNetB0 : 0.4 | ResNet50 : 0.3 | YOLOv8 : 0.3
probs_fusion = (0.4 * all_probs_eff + 
                0.3 * all_probs_res + 
                0.3 * all_probs_yolo)

# Mapping classe → normal/anormal
ANORMAL_IDX = [CLASSES.index(c) for c in ANORMAL_CLASSES if c in CLASSES]
NORMAL_IDX  = [CLASSES.index(c) for c in NORMAL_CLASSES if c in CLASSES]

# Score anormal = somme des probabilités des classes anormales
score_anormal = probs_fusion[:, ANORMAL_IDX].sum(axis=1)
score_normal  = probs_fusion[:, NORMAL_IDX].sum(axis=1)

# Décision finale
predictions = (score_anormal >= 0.5).astype(int)

# Labels réels
true_labels = np.array([1 if CLASSES[l] in ANORMAL_CLASSES else 0 
                         for l in all_labels])

# Métriques
from sklearn.metrics import accuracy_score, recall_score, classification_report

acc    = accuracy_score(true_labels, predictions)
recall = recall_score(true_labels, predictions)

print(f"✅ Late Fusion Comportement")
print(f"Accuracy  : {acc*100:.1f}%")
print(f"Recall anormal : {recall*100:.1f}%")
print(f"\n{classification_report(true_labels, predictions, target_names=['normal','anormal'])}")

✅ Late Fusion Comportement
Accuracy  : 99.8%
Recall anormal : 100.0%

              precision    recall  f1-score   support

      normal       1.00      1.00      1.00       320
     anormal       1.00      1.00      1.00       320

    accuracy                           1.00       640
   macro avg       1.00      1.00      1.00       640
weighted avg       1.00      1.00      1.00       640



La capacité du système à distinguer un comportement normal d'un comportement anormal chez un chien, en combinant les 3 modèles.


Recall anormal 100% → le plus important pour notre projet :
Sur tous les chiens présentant des comportements dangereux → aucun n'a été raté → aucun chien à risque de rage n'a été classifié comme normal.
C'est la métrique critique pour un système de détection de rage → mieux vaut une fausse alarme qu'un chien dangereux non détecté.


In [29]:
import numpy as np

np.save("/kaggle/working/behav_probs_fusion.npy", probs_fusion)
np.save("/kaggle/working/behav_true_labels.npy", true_labels)
print("✅ Probabilités sauvegardées")

✅ Probabilités sauvegardées


In [30]:
import os, shutil

# Créer dossier output
os.makedirs("/kaggle/working/late_fusion_data", exist_ok=True)

# Copier les fichiers
shutil.copy("/kaggle/working/behav_probs_fusion.npy", 
            "/kaggle/working/late_fusion_data/behav_probs_fusion.npy")
shutil.copy("/kaggle/working/behav_true_labels.npy", 
            "/kaggle/working/late_fusion_data/behav_true_labels.npy")

print(os.listdir("/kaggle/working/late_fusion_data"))
print("✅ Fichiers prêts")

['behav_probs_fusion.npy', 'behav_true_labels.npy']
✅ Fichiers prêts
